# 03 — Sampling: temperature, top_p, top_k

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase02/notebooks/03_sampling_params.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivo.** Tocar los parámetros de sampling y ver cómo cambia el output. La idea es que después puedas elegirlos a conciencia para tu caso de uso.

**Requisitos.** API key de Groq en `GROQ_API_KEY`.


In [1]:
%pip install --quiet groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.5 MB/s eta 0:00:00


In [11]:
import os
from groq import Groq

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    assert os.environ.get("GROQ_API_KEY"), "Exportá GROQ_API_KEY."

client = Groq()
MODELS = {
    "llama_fast":   "llama-3.1-8b-instant",
    "llama_strong": "llama-3.3-70b-versatile",
    "qwen_reason":  "qwen/qwen3-32b",
    "deepseek":     "deepseek-r1-distill-llama-70b",
    "gemma":        "gemma2-9b-it",
}
MODEL = MODELS["llama_strong"]  # cambiá la clave para probar otros modelos

def generar(prompt, **kwargs):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
    )
    return resp.choices[0].message.content


## 1. `temperature` — el termostato de la creatividad

Mismo prompt, distintas temperaturas. Observá la diferencia de tono y vocabulario.


In [12]:
PROMPT = "Escribime un poema corto (4 versos) sobre el otoño en Buenos Aires."

for temp in [0.0, 0.3, 0.7, 1.2]:
    print(f"--- temperature = {temp} ---")
    print(generar(PROMPT, temperature=temp))
    print()


--- temperature = 0.0 ---
En Buenos Aires, el otoño llega suave,
Con vientos frescos y un sol que se esconde.
Las hojas caen, doradas y rojizas, 
Y la ciudad se viste de un manto de oro.

--- temperature = 0.3 ---
En Buenos Aires, el otoño llega suave,
Con vientos frescos y un cielo que se deslava.
Las hojas caen, doradas y rojizas, al suelo,
Y la ciudad se viste de un manto de otoño melancólico y sereno.

--- temperature = 0.7 ---
En Buenos Aires, el otoño llega suave,
Con vientos frescos y un sol más grave.
Las hojas caen de los árboles altos,
Y la ciudad se viste de colores dorados.

--- temperature = 1.2 ---
En Buenos Aires, el otoño llega suave,
Con vientos frescos y una luz que se evade.
Las hojas caen, y un manto de dorado,
Cubre las calles, en un cálido y sensual abrazado.



- `temperature=0` → el modelo elige siempre el token más probable. Determinista.
- Valores altos → distribución más plana, salidas diversas (y a veces incoherentes).


## 2. Reproducibilidad: el bug del "siempre lo mismo"

Con temperatura baja, dos llamadas seguidas dan respuestas casi idénticas. Útil cuando necesitás determinismo (testing, código).


In [13]:
PROMPT = "Listame 3 razones por las que se prefiere PostgreSQL sobre MySQL."

for i in range(2):
    print(f"--- Llamada {i+1} (temperature=0) ---")
    print(generar(PROMPT, temperature=0))
    print()


--- Llamada 1 (temperature=0) ---
¡Claro! A continuación, te presento 3 razones por las que se prefiere PostgreSQL sobre MySQL:

1. **Soporte para tipos de datos avanzados**: PostgreSQL ofrece un conjunto más amplio de tipos de datos, incluyendo tipos de datos espaciales, temporales y de arrays, lo que lo hace más versátil y flexible para manejar datos complejos. Por ejemplo, PostgreSQL admite tipos de datos como `geometry`, `geography` y `interval`, que no están disponibles en MySQL.

2. **Integridad referencial y transacciones**: PostgreSQL tiene un soporte más robusto para la integridad referencial y las transacciones, lo que garantiza la consistencia y la coherencia de los datos. PostgreSQL admite transacciones anidadas, transacciones paralelas y bloqueos de fila, lo que lo hace más adecuado para aplicaciones que requieren un alto nivel de concurrencia y seguridad.

3. **Extensibilidad y personalización**: PostgreSQL es altamente extensible y personalizable, lo que permite a los de

## 3. `top_p` — nucleus sampling

Muestrea solo del conjunto de tokens cuya probabilidad acumulada llega a P. Recorta la "cola larga".


In [10]:
PROMPT = "Inventame el nombre de una banda de rock progresivo argentina."

for p in [0.0]:
    print(f"--- top_p = {p} ---")
    for _ in range(1):
        print("  ·", generar(PROMPT, temperature=0.0, top_p=p))
    print()


--- top_p = 0.0 ---
  · <think>
Okay, the user wants me to invent a name for a progressive rock band from Argentina. Let me start by thinking about the key elements here. Progressive rock is known for its complex compositions, experimental sounds, and often conceptual themes. The band needs to have an Argentine identity, so maybe incorporating elements from the culture, language, or geography of Argentina.

First, I should consider Spanish words related to Argentina. Words like "Andes" (the mountain range), "Patagonia" (the southern region), "tango" (the dance/music), or "gauchos" (traditional cowboys). Also, maybe using indigenous languages like Quechua or Mapudungun for a unique touch. For example, "Araucano" is a term from the Mapuche people.

Next, the name should sound musical and have a certain rhythm. Progressive rock bands often have names that are a bit abstract or poetic. Maybe combining two words or using a metaphor. Words related to time, space, or transformation could work

## Cuándo usar qué

| Caso | temperature | top_p |
|---|---|---|
| Código, factual, extracción | 0.0 – 0.3 | 1.0 |
| Conversación natural | 0.6 – 0.8 | 0.9 |
| Creatividad, brainstorming | 0.9 – 1.2 | 0.95 |
| Determinismo (tests) | 0.0 | 1.0 |
